In [4]:
!pip3 install joblib scikit-learn xgboost
# If '!pip3 install joblib scikit-learn xgboost' throuhgs an error then try any of following two
#  !pip install joblib scikit-learn xgboost
# !python -m pip install joblib scikit-learn xgboost

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Frameworks/Python.framework/Versions/3.8/bin/python3.8 -m pip install --upgrade pip' command.


In [5]:
    import joblib
    import json
    import numpy as np
    import pandas as pd
    from sklearn.metrics import mean_squared_error, r2_score

    # Load processed test data (full 15% held-out split, 734K rows)
    test_df = pd.read_parquet('data/processed/test.parquet')
    with open('data/processed/feature_info.json', 'r') as f:
        feature_info = json.load(f)

    X_test = test_df[feature_info['all_features']]
    y_test = test_df['responder_6']
    w_test = test_df['weight']
    print(f"Test set loaded: {len(X_test):,} rows, {X_test.shape[1]} features")

    # Load all saved models
    train_mean = joblib.load('models/trivial_mean.pkl')
    ridge_model = joblib.load('models/ridge_model.pkl')
    ridge_scaler = joblib.load('models/ridge_scaler.pkl')
    rf_model = joblib.load('models/rf_model.pkl')
    xgb_model = joblib.load('models/xgb_model.pkl')
    xgbt_model = joblib.load('models/xgbt_model.pkl')
    stack_model = joblib.load('models/stack_model.pkl')
    stack_scaler = joblib.load('models/stack_scaler.pkl')
    print("All models loaded successfully")

Test set loaded: 733,897 rows, 79 features


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Ridge from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.4.2 when using version 1.8.0. This might lead to breakin

All models loaded successfully


In [6]:

    X_test_dedup = X_test.loc[:, ~X_test.columns.duplicated()]
    results = {}

    #Trivial Baseline — predict training mean
    triv_pred = np.full(len(y_test), train_mean)
    results['Trivial (mean)'] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, triv_pred, sample_weight=w_test)),
        'R^2': r2_score(y_test, triv_pred, sample_weight=w_test)
    }

    #Ridge Regression — requires scaling
    ridge_pred = ridge_model.predict(ridge_scaler.transform(X_test))
    results['Ridge Regression'] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, ridge_pred, sample_weight=w_test)),
        'R^2': r2_score(y_test, ridge_pred, sample_weight=w_test)
    }

    #Random Forest
    rf_pred = rf_model.predict(X_test)
    results['Random Forest'] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, rf_pred, sample_weight=w_test)),
        'R^2': r2_score(y_test, rf_pred, sample_weight=w_test)
    }

    # XGBoost Default
    xgb_pred = xgb_model.predict(X_test_dedup.values)
    results['XGBoost (Default)'] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, xgb_pred, sample_weight=w_test)),
        'R^2': r2_score(y_test, xgb_pred, sample_weight=w_test)
    }

    #XGBoost Tuned
    xgbt_pred = xgbt_model.predict(X_test_dedup.values)
    results['XGBoost (Tuned)'] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, xgbt_pred, sample_weight=w_test)),
        'R^2': r2_score(y_test, xgbt_pred, sample_weight=w_test)
    }

    #Stacking Ensemble — Ridge + XGBoost Tuned
    meta_input = np.column_stack([ridge_pred, xgbt_pred])
    stack_pred = stack_model.predict(stack_scaler.transform(meta_input))
    results['Stacking Ensemble'] = {
        'RMSE': np.sqrt(mean_squared_error(y_test, stack_pred, sample_weight=w_test)),
        'R^2': r2_score(y_test, stack_pred, sample_weight=w_test)
    }

    # Print all results
    print(f"\n{'Model':<25} {'Test RMSE':>10} {'Test R^2':>10}")
    print("-" * 50)
    for name, scores in results.items():
        print(f"{name:<25} {scores['RMSE']:>10.4f} {scores['R^2']:>10.4f}")

    best = max(results.items(), key=lambda x: x[1]['R^2'])
    triv_rmse = results['Trivial (mean)']['RMSE']
    improvement = (triv_rmse - best[1]['RMSE']) / triv_rmse * 100
    print(f"\nBest: {best[0]} — RMSE {best[1]['RMSE']:.4f}, R^2 {best[1]['R^2']:.4f}")
    print(f"{improvement:.1f}% RMSE improvement over trivial baseline")


Model                      Test RMSE   Test R^2
--------------------------------------------------
Trivial (mean)                0.7689    -0.0000
Ridge Regression              0.3031     0.8446
Random Forest                 0.3020     0.8457
XGBoost (Default)             0.2986     0.8492
XGBoost (Tuned)               0.2971     0.8507
Stacking Ensemble             0.2964     0.8514

Best: Stacking Ensemble — RMSE 0.2964, R^2 0.8514
61.5% RMSE improvement over trivial baseline
